# Huấn Luyện Benchmark Mô Hình UC-TTT Trên Google Colab
## (User-Credibility-Aware Temporal Tree Transformer vs Original TTT)

Notebook này được thiết kế để huấn luyện mô hình rumor detection với cấu hình đồng bộ 100% theo bài báo gốc **Wu et al. (PLOS ONE 2025)** trên GPU đám mây của Google Colab (Tesla T4 / A100).

### Các bước thực hiện:
1. **Bật GPU:** Vào `Runtime` -> `Change runtime type` -> Chọn **T4 GPU** (hoặc A100 nếu có Colab Pro).
2. **Mount Google Drive:** Để lưu checkpoint và copy 2 file data đã xử lý (`pheme_processed.pkl` và `text_embeddings.pt`).
3. **Clone Repo & Cài đặt Thư viện**.
4. **Chạy Thực Nghiệm Đối Chứng:** 1 lần TTT Baseline (tắt UC) và 1 lần UC-TTT Proposed (bật UC).
5. **Xuất Bảng So Sánh Hiệu Năng Chi Tiết**.

In [ ]:
# Bước 1: Kiểm tra cấu hình GPU được cấp phát
!nvidia-smi

In [ ]:
# Bước 2: Kết nối Google Drive để nạp dữ liệu và lưu checkpoint an toàn
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Bước 3: Clone Codebase từ GitHub
%cd /content
!git clone https://github.com/tphu2029/SNA_EnhancingTTT-via-User-Credibility-Integration-for-Social-Media-Rumor-Detection.git repo
%cd /content/repo
!git checkout feat/issue-4-ucttt-proposed
!git pull

In [ ]:
# Bước 4: Cài đặt các thư viện phụ thuộc
!pip install -q -r requirements.txt

### Chuẩn bị dữ liệu trên Google Drive
Hãy tải 2 file dữ liệu đã xử lý từ máy của bạn lên thư mục Google Drive (ví dụ trong thư mục `My Drive/PHEME_DATA/`):
1. `pheme_processed.pkl` (~181 MB)
2. `text_embeddings.pt` (~192 MB)

Sau đó chạy ô bên dưới để copy vào thư mục `data/processed/` của project.

In [ ]:
import os
import shutil

# Đường dẫn thư mục chứa data trên Google Drive của bạn (tùy chỉnh nếu đặt tên khác)
DRIVE_DATA_DIR = "/content/drive/MyDrive/PHEME_DATA"

os.makedirs("data/processed", exist_ok=True)

for filename in ["pheme_processed.pkl", "text_embeddings.pt"]:
    src = os.path.join(DRIVE_DATA_DIR, filename)
    dst = os.path.join("data/processed", filename)
    if os.path.exists(src):
        print(f"Đang copy {filename} từ Google Drive...")
        shutil.copyfile(src, dst)
        print(f"-> Hoàn tất: {dst} ({os.path.getsize(dst) / 1024 / 1024:.1f} MB)")
    else:
        print(f"[CẢNH BÁO] Không tìm thấy file {src}. Vui lòng kiểm tra lại đường dẫn trên Drive!")

### Bước 5: Chạy Toàn Bộ Benchmark Đồng Bộ Bài Báo (TTT vs UC-TTT)
Script dưới đây sẽ tự động thực hiện:
1. **Lần 1:** Train TTT Baseline (Wu et al. 2025) trên 5 sự kiện LOEO (tắt UC).
2. **Lần 2:** Train UC-TTT Proposed trên 5 sự kiện LOEO (bật UC với Gated Fusion).
3. **Tổng hợp:** Xuất bảng so sánh delta trực tiếp.

In [ ]:
# Chạy tự động cả 2 mô hình (TTT Baseline và UC-TTT) với kiến trúc chuẩn bài báo 768-d 6 layers
!python -u experiments/run_paper_sync_benchmarks.py

### Bước 6: Lưu Kết Quả & Checkpoints Về Google Drive

In [ ]:
# Sao lưu toàn bộ file kết quả JSON và bảng so sánh về Google Drive
BACKUP_DIR = "/content/drive/MyDrive/PHEME_DATA/results_backup"
os.makedirs(BACKUP_DIR, exist_ok=True)

!cp -r results/* "$BACKUP_DIR/"
print(f"Đã sao lưu toàn bộ kết quả vào: {BACKUP_DIR}")